# Sandbox: Yair

Conflict, Democracy and Arms project. This is your personal notebook for trying things out.
Only Yair edits this file, so it never causes a merge conflict with Nicolas.

**How every task works**

1. **What you need:** each new command is explained in one or two sentences, followed by a small example on made-up data. Run the example and look at the output. Change a number in it and run it again to see what happens.
2. **Your turn:** the same idea on our project data. Every step has its own empty cell. You write the code, using the example as a model.
3. **Hints:** only if you are stuck.
4. **Checkpoint:** a test cell. OK means the task is done.

When a task is done, paste the working code into the main notebook listed below, then commit and push.

| Part | Task | Paste into |
| --- | --- | --- |
| Practice | Merging on toy data | nothing, just for learning |
| A | SIPRI: arms imports per country-year and top suppliers | `04_sipri.ipynb` |
| B | Descriptives: top importers, imports by regime, democracy in war and peace | `06_analysis.ipynb`, Part A |
| C | Chart functions: arms imports, suppliers, global trends, democracy line, main chart | `07_charts.ipynb`, then `app/panels_arms.py` (C1 to C3) and `app/panels_analysis.py` (C4, C5) |

Variable, column and file names come from `CONVENTIONS.md`. Use them exactly.

## Command card

Everything you need in this notebook, with the lecture where you saw it. Come back here when you forget a command.

| Command | What it does | Lecture |
| --- | --- | --- |
| `pd.read_csv(path)`, `pd.read_excel(path)` | load a file into a table (DataFrame) | week 3, week 5 joins |
| `df.head()`, `df.shape`, `df.columns`, `df.info()` | look at a table | week 3 |
| `df["col"]` | one column | week 3 |
| `df[df["year"] >= 1950]` | keep rows where a condition is True | week 3 cleaning |
| `cond1 & cond2`, `~cond` | both conditions, "not" | week 3 cleaning |
| `df["col"].between(a, b)`, `df["col"].isin([...])` | conditions for ranges and lists | week 3, week 4 |
| `df.rename(columns={"old": "new"})` | rename columns | week 3 |
| `df[["a", "b"]]` | keep only some columns | week 3 |
| `df.dropna(subset=["col"])`, `df["col"].isna()` | missing values | week 3 cleaning |
| `df["col"].astype(int)` | change the type | week 3 cleaning |
| `df.drop_duplicates()`, `df.duplicated()` | duplicates | week 3, week 5 joins |
| `df.sort_values(["a", "b"])` | sort | week 3 |
| `df.head(n)`, `df["col"].unique()`, `df["col"].value_counts()` | first rows, distinct values, counts per value | week 3, week 4 |
| `df["col"].fillna(0)` | replace missing values | week 3 cleaning |
| `df["col"].map(dictionary)` | translate values with a dictionary | week 4 Disney |
| `df["col"].str.endswith("*")` | True where the text ends with `*` | new, works like `str.contains` |
| `df.drop(columns="col")` | remove a column | week 3 |
| `pd.concat([df1, df2])` | stack tables | week 5 joins |
| `text.split(",")`, `text.strip()` | split or clean one piece of text (plain Python) | week 2 |
| `df["col"].str.strip()`, `df["col"].str.contains("x")` | the same for a whole column | week 3 cleaning |
| `for x in my_list:` | loop | week 2 |
| `for a, b in zip(df["a"], df["b"]):` | loop over two columns at once | week 4 |
| `my_list.append(x)` | add to a list | week 2 |
| `d = {}`, `d[key] = value`, `d.get(key)` | dictionary | week 2 |
| `pd.DataFrame(list_of_dicts)` | turn a list of dictionaries into a table | week 3 |
| `df.groupby("a").agg(n=("b", "count"))` | one row per group with summaries; functions: `"count"`, `"nunique"` (distinct values), `"sum"`, `"mean"`, `"max"` | week 4 Disney |
| `df.groupby("a")["b"].transform("mean")` | group value repeated on every row | week 4 Disney |
| `for name, group in df.groupby("a"):` | loop over groups | week 5 data viz |
| `pd.merge(left, right, on=[...], how="left", validate=..., indicator=True)` | join two tables | week 5 joins |
| `df.pivot(index=..., columns=..., values=...)` | long to wide | week 4 |
| `df.to_csv(path, index=False)` | save a table | week 3 |
| `fig, ax = plt.subplots()`, `ax.plot(x, y)`, `ax.bar(...)`, `ax.barh(...)` | charts | week 5 data viz |

If you see a `SettingWithCopyWarning`, add `.copy()` after the filter that made the table, for example `df = panel[panel["iso3"] == "UKR"].copy()`.

## Setup

This cell loads the packages and sets the folder paths. Run it every time you open the notebook.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAW = "../data/raw/"
CLEAN = "../data/clean/"

pd.set_option("display.max_columns", 50)
print("pandas", pd.__version__)

---
## Practice: merging (do this before any real merge)

We never merge big tables without testing on tiny ones first.

#### What you need: merging two tables

A merge puts two tables side by side. Rows are matched by the value in one or more key columns.

- `on=` names the key column(s); for two keys, write a list: `on=["iso3", "year"]`.
- `how="inner"` keeps only keys found in both tables; `how="left"` keeps every row of the left table and leaves gaps (NaN) where the right table has nothing.
- `validate="one_to_one"` or `"many_to_one"` makes pandas stop with an error if a key appears more often than it should. This protects you from rows multiplying by accident.
- `indicator=True` adds a column `_merge` that says for every row whether it was found in both tables (`both`) or only in the left one (`left_only`).

*Seen in the lecture:* week 5, page `lecture_week_5_joins.py` (sections on left joins, `validate` and `indicator`)

In [ ]:
# Example: run this cell and look at the output
students = pd.DataFrame({"name": ["Anna", "Ben", "Cara"], "age": [21, 23, 22]})
grades = pd.DataFrame({"name": ["Ben", "Cara", "Dan"], "grade": [5.5, 4.0, 6.0]})

both = pd.merge(students, grades, on="name", how="inner")
print("inner:")
print(both)

everyone = pd.merge(students, grades, on="name", how="left", validate="one_to_one", indicator=True)
print("left:")
print(everyone)
print(everyone["_merge"].value_counts())

#### Your turn

Run the cell below. It creates two small tables from our project world.

In [ ]:
demo = pd.DataFrame({
    "iso3": ["CHE", "IND", "UKR"],
    "year": [2000, 2000, 2000],
    "democracy_index": [0.90, 0.70, 0.45],
})
arms = pd.DataFrame({
    "iso3": ["IND", "UKR", "SAU"],
    "year": [2000, 2000, 2000],
    "arms_imports_tiv": [900, 20, 1500],
})
print(demo)
print(arms)

**Step 1.** Predict first, in a comment: which countries appear in both tables? Which only in one?

In [ ]:
# your code here

**Step 2.** Inner merge of `demo` and `arms` on `["iso3", "year"]`. Call it `m_inner` and print it. Which country is missing, and why?

In [ ]:
# your code here

**Step 3.** Left merge with `demo` on the left, `validate="one_to_one"` and `indicator=True`. Call it `m_left` and print it. What is in `arms_imports_tiv` for CHE, and what does `_merge` say?

In [ ]:
# your code here

**Step 4.** In our project, no arms delivery means zero imports. Replace the missing value: `m_left["arms_imports_tiv"] = m_left["arms_imports_tiv"].fillna(0)`. Print again.

In [ ]:
# your code here

In [ ]:
arms_dup = pd.concat([arms, pd.DataFrame({"iso3": ["IND"], "year": [2000], "arms_imports_tiv": [5]})])
print(arms_dup)

**Step 5.** Run the cell above first, it makes `arms_dup` with India twice. Then try a left merge of `demo` with `arms_dup` with `validate="one_to_one"`. Read the error message. Then do it without `validate`, call it `m_dup`, and count the rows. What happened to India?

In [ ]:
# your code here

**Step 6.** Write two rules in your own words (as a comment): what does `how="left"` keep, and what does `validate` protect you from?

In [ ]:
# your rules here

**Checkpoint.**

In [ ]:
assert len(m_inner) == 2, "inner keeps only countries in both tables"
assert len(m_left) == 3, "left keeps every row of demo"
assert sorted(m_left["iso3"]) == ["CHE", "IND", "UKR"], "a left merge keeps the rows of the LEFT table (demo)"
assert "_merge" in m_left.columns, "use indicator=True"
assert m_left["arms_imports_tiv"].isna().sum() == 0, "fill the missing value with 0"
assert len(m_dup) == 4, "the duplicate India row in arms_dup creates an extra row"
print("OK")

---
## Part A: SIPRI (04)

Needs `crosswalk_sipri.csv` from 01 Crosswalk for A4.

### A1 Load the trade register

**Goal:** `sipri_raw` loaded correctly.

#### What you need: files with notes on top and special characters

Some files start with a few lines of notes before the column names. `skiprows=11` tells `read_csv` to skip the first 11 lines. Files can also be saved in different character encodings; this one uses Latin-1, so we add `encoding="latin-1"`. Without it, names like "Côte d'Ivoire" cause a `UnicodeDecodeError`.

*Seen in the lecture:* reading files and `encoding`: week 4 fundamentals; `skiprows` is new (pandas docs: `read_csv`)

In [ ]:
# Example: run this cell and look at the output
text = "Notes from the shop\nCreated 2024\n\nproduct,price\nCafé,4.5\nTea,3.0\n"
with open("example_notes.csv", "w", encoding="latin-1") as f:     # make a small file with 3 note lines
    f.write(text)

print(pd.read_csv("example_notes.csv", skiprows=3, encoding="latin-1"))

#### Your turn

**Step 1.** Open `trade-register.csv` in a text editor (TextEdit or VS Code, not Excel) and look at the first 15 lines. On which line do the column names start? Write it in a comment.

In [ ]:
# your code here

**Step 2.** Load it with `pd.read_csv(RAW + "trade-register.csv", skiprows=11, encoding="latin-1")` into `sipri_raw`. Print shape, column names and the first rows.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert sipri_raw.shape == (62009, 17), f"expected (62009, 17), got {sipri_raw.shape}"
for col in ["Supplier", "Recipient", "Delivery year", "TIV delivery values"]:
    assert col in sipri_raw.columns, f"missing column {col}"
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### A2 Rename and keep what you need

**Goal:** `sipri` with exactly the columns `supplier`, `recipient`, `year`, `tiv`, `category`, `status`, and recipient names without extra spaces.

#### What you need: renaming, choosing columns, cleaning text

`rename(columns={...})` renames; `df[list_of_names]` keeps those columns in that order; `df["col"].str.strip()` removes spaces at the start and end of every text in a column.

*Seen in the lecture:* week 3, `02_data_cleaning_full.ipynb`

In [ ]:
# Example: run this cell and look at the output
shop = pd.DataFrame({"Product Name": [" Tea", "Coffee "], "Price CHF": [3.0, 4.5], "Shelf": [1, 2]})
shop2 = shop.rename(columns={"Product Name": "product", "Price CHF": "price"})
shop2 = shop2[["product", "price"]].copy()
shop2["product"] = shop2["product"].str.strip()
print(shop2)
print(list(shop2["product"]))

#### Your turn

**Step 1.** Write a dictionary `new_names`: `Supplier` → `supplier`, `Recipient` → `recipient`, `Delivery year` → `year`, `TIV delivery values` → `tiv`, `Armament category` → `category`, `Status` → `status`. Rename and call the result `sipri_raw_renamed`.

In [ ]:
# your code here

**Step 2.** Keep only the six new columns, in this order: `sipri = sipri_raw_renamed[keep].copy()` with a list `keep`.

In [ ]:
# your code here

**Step 3.** Remove spaces around the recipient names with `.str.strip()`. 01 Crosswalk does the same, so the names match in A4.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(sipri.columns) == ["supplier", "recipient", "year", "tiv", "category", "status"]
assert len(sipri) == 62009, "no rows should be lost in this step"
assert (sipri["recipient"] == sipri["recipient"].str.strip()).all(), "strip the recipient names"
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### A3 Drop undelivered deals and non-states

**Goal:** only delivered arms to states. Write down after each step how many rows were removed.

#### What you need: removing rows step by step

- `df.dropna(subset=["col"])` removes rows with a missing value in that column.
- `df["col"].str.endswith("*")` is True for texts that end with a star. `~` turns True into False and back, so `df[~condition]` keeps the other rows.
- `df[df["col"] != "x"]` keeps rows that are not equal to "x".

*Seen in the lecture:* week 3, `02_data_cleaning_full.ipynb` (missing values, conditions)

In [ ]:
# Example: run this cell and look at the output
pets = pd.DataFrame({"name": ["Rex", "Mia*", "Tom", "unknown", None], "age": [3, 5, None, 2, None]})
before = len(pets)
pets = pets.dropna(subset=["age"])
print(before - len(pets), "rows removed (no age)")

before = len(pets)
pets = pets[~pets["name"].str.endswith("*")]
pets = pets[pets["name"] != "unknown"]
print(before - len(pets), "rows removed (names)")
print(pets)

#### Your turn

**Step 1.** Count rows with a missing `year`. It should be 1,220: deals ordered but not delivered by 2025. Remove them with `dropna`. Print the count before and after.

In [ ]:
# your code here

**Step 2.** Make `year` a whole number with `astype(int)`.

In [ ]:
# your code here

**Step 3.** Print the unique recipient names that end with a star. One star = non-state actor, two stars = organisation (a name ending in `**` also ends in `*`). Then keep only the other rows.

In [ ]:
# your code here

**Step 4.** Print the names that contain "unknown" (`str.contains("unknown")`) to see the exact spelling, and remove those rows too. Write all removed numbers in a comment.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert sipri["year"].dtype.kind == "i"
assert not sipri["recipient"].str.endswith("*").any()
assert sipri["recipient"].str.contains("unknown").sum() == 0
assert 50000 < len(sipri) <= 62009 - 1220, "too many or too few rows removed"
print("rows left:", len(sipri))
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### A4 Add iso3 with a merge

**Goal:** every remaining row has an `iso3`; rows that cannot be matched are removed on purpose and written down.

#### What you need: merging two tables

A merge puts two tables side by side. Rows are matched by the value in one or more key columns.

- `on=` names the key column(s); for two keys, write a list: `on=["iso3", "year"]`.
- `how="inner"` keeps only keys found in both tables; `how="left"` keeps every row of the left table and leaves gaps (NaN) where the right table has nothing.
- `validate="one_to_one"` or `"many_to_one"` makes pandas stop with an error if a key appears more often than it should. This protects you from rows multiplying by accident.
- `indicator=True` adds a column `_merge` that says for every row whether it was found in both tables (`both`) or only in the left one (`left_only`).

*Seen in the lecture:* week 5, page `lecture_week_5_joins.py` (sections on left joins, `validate` and `indicator`)

In [ ]:
# Example: run this cell and look at the output
students = pd.DataFrame({"name": ["Anna", "Ben", "Cara"], "age": [21, 23, 22]})
grades = pd.DataFrame({"name": ["Ben", "Cara", "Dan"], "grade": [5.5, 4.0, 6.0]})

both = pd.merge(students, grades, on="name", how="inner")
print("inner:")
print(both)

everyone = pd.merge(students, grades, on="name", how="left", validate="one_to_one", indicator=True)
print("left:")
print(everyone)
print(everyone["_merge"].value_counts())

#### Your turn

**Step 1.** Load `CLEAN + "crosswalk_sipri.csv"` into `crosswalk_sipri`. Print the first rows. The key is `recipient`. Save `rows_before_merge = len(sipri)`.

In [ ]:
# your code here

**Step 2.** Left merge the crosswalk onto `sipri` on `recipient` with `validate="many_to_one"` and `indicator=True`. Overwrite `sipri`. Print the length; it must equal `rows_before_merge`.

In [ ]:
# your code here

**Step 3.** Print `sipri["_merge"].value_counts()` and the unique recipient names with `left_only`. Most should be fixed in the crosswalk (01): historical states map to the V-Dem unit of those years.

In [ ]:
# your code here

**Step 4.** For names that really have no V-Dem country: print how many rows and how much TIV they have, write that in the decisions table, then keep only the rows with `_merge == "both"` and drop the `_merge` column.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert "iso3" in sipri.columns and "_merge" not in sipri.columns
assert sipri["iso3"].isna().sum() == 0, "remove the unmatched rows in step 4 (and write them down)"
assert len(sipri) <= rows_before_merge
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### A5 Imports per country-year

**Goal:** `sipri_cy` with exactly the columns `iso3`, `year`, `arms_imports_tiv`.

#### What you need: adding up per group

`df.groupby(["a", "b"], as_index=False).agg(total=("x", "sum"))` adds up `x` for every combination of `a` and `b`.

*Seen in the lecture:* week 4, `disney_cleaning_analysis.ipynb`

In [ ]:
# Example: run this cell and look at the output
orders = pd.DataFrame({"shop": ["A", "A", "A", "B"], "day": [1, 1, 2, 1], "amount": [10.5, 4.0, 7.0, 3.0]})
a_day1 = orders[(orders["shop"] == "A") & (orders["day"] == 1)]
print("by hand:", a_day1["amount"].sum())
print(orders.groupby(["shop", "day"], as_index=False).agg(total=("amount", "sum")))

#### Your turn

**Step 1.** Check one case by hand: filter India in 2000 and add up `tiv`. Store it in `india_2000_hand`.

In [ ]:
# your code here

**Step 2.** Group by `["iso3", "year"]` with `as_index=False` and `agg(arms_imports_tiv=("tiv", "sum"))`. Call it `sipri_cy`.

In [ ]:
# your code here

**Step 3.** Compare India 2000 in `sipri_cy` with your hand number.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

A year without deliveries does not appear in `sipri_cy`. It becomes 0 in the merge (05), like CHE in the practice.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(sipri_cy.columns) == ["iso3", "year", "arms_imports_tiv"]
assert sipri_cy.duplicated(["iso3", "year"]).sum() == 0
assert (sipri_cy["arms_imports_tiv"] >= 0).all()
india = sipri_cy[(sipri_cy["iso3"] == "IND") & (sipri_cy["year"] == 2000)]["arms_imports_tiv"].sum()
assert abs(india - india_2000_hand) < 0.001, "India 2000 differs from your hand calculation"
assert abs(sipri_cy["arms_imports_tiv"].sum() - sipri["tiv"].sum()) < 0.01, "the total TIV must not change"
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### A6 Top suppliers and save

**Goal:** `sipri_suppliers` with exactly the columns `iso3`, `year`, `supplier`, `tiv`, where only the top 3 suppliers of each country keep their name and the rest is called "Other".

#### What you need: the top entries per group, and replacing the rest

Three small steps: (1) total per group with `groupby` + `agg`, then `sort_values` and `head(3)`; (2) store the top names of every group in a dictionary of lists; (3) a loop that keeps a name if it is in the list of its group, otherwise writes "Other".

*Seen in the lecture:* loops over groups: week 5 data viz; dictionaries and `zip`: week 4

In [ ]:
# Example: run this cell and look at the output
sales = pd.DataFrame({
    "shop": ["A", "A", "A", "A", "B", "B"],
    "product": ["tea", "coffee", "cake", "juice", "tea", "cake"],
    "amount": [5, 9, 2, 1, 4, 6],
})

top2 = {}
for shop, rows in sales.groupby("shop"):
    totals = rows.groupby("product", as_index=False).agg(total=("amount", "sum"))
    totals = totals.sort_values("total", ascending=False)
    top2[shop] = list(totals.head(2)["product"])
print(top2)

grouped_names = []
for shop, product in zip(sales["shop"], sales["product"]):
    if product in top2[shop]:
        grouped_names.append(product)
    else:
        grouped_names.append("Other")
sales["product_group"] = grouped_names
print(sales)

#### Your turn

**Step 1.** India first: filter India, total `tiv` per `supplier`, sort, `head(3)`, and turn the names into a list. Print it.

In [ ]:
# your code here

**Step 2.** For all countries, like the example: a dictionary `top3` with a list of the three biggest suppliers for every `iso3`. Print `top3["IND"]`.

In [ ]:
# your code here

**Step 3.** A column `supplier_group`: the supplier's name if it is in the top 3 of its country, otherwise `"Other"`.

In [ ]:
# your code here

**Step 4.** Total `tiv` per `iso3`, `year`, `supplier_group` with `groupby(..., as_index=False).agg(tiv=("tiv", "sum"))`, then rename `supplier_group` to `supplier`. Call it `sipri_suppliers`.

In [ ]:
# your code here

**Step 5.** Save `sipri_cy.csv` and `sipri_suppliers.csv` with `index=False`.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(sipri_suppliers.columns) == ["iso3", "year", "supplier", "tiv"]
assert sipri_suppliers.duplicated(["iso3", "year", "supplier"]).sum() == 0
assert sipri_suppliers.groupby("iso3")["supplier"].nunique().max() <= 4, "top 3 + Other"
assert set(sipri_suppliers[sipri_suppliers["iso3"] == "IND"]["supplier"]) <= set(top3["IND"] + ["Other"])
assert abs(sipri_suppliers["tiv"].sum() - sipri_cy["arms_imports_tiv"].sum()) < 0.01, "the total TIV must not change"
assert os.path.exists(CLEAN + "sipri_cy.csv") and os.path.exists(CLEAN + "sipri_suppliers.csv")
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

---
## Part B: Descriptives (06 Part A)

Needs `panel.csv` from 05 Merge.

### B1 Top 15 importers

**Goal:** `top15`: the 15 countries with the highest total TIV since 1950.

#### What you need: totals, sorting and the first rows

`groupby` + `agg` gives totals; `sort_values("col", ascending=False)` puts the largest first; `head(15)` keeps the first 15 rows.

*Seen in the lecture:* week 4 Disney notebook

In [ ]:
# Example: run this cell and look at the output
sales = pd.DataFrame({"shop": ["A", "B", "C", "A", "B"], "amount": [5, 9, 2, 1, 4]})
totals = sales.groupby("shop", as_index=False).agg(total=("amount", "sum"))
print(totals.sort_values("total", ascending=False).head(2))

#### Your turn

**Step 1.** Load `CLEAN + "panel.csv"` into `panel`.

In [ ]:
# your code here

**Step 2.** Total `arms_imports_tiv` per `country_name`, call it `totals`.

In [ ]:
# your code here

**Step 3.** Sort from large to small, keep the first 15, call it `top15` and print it. Any surprises? One sentence as a comment.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(top15.columns) == ["country_name", "arms_imports_tiv"], "use agg(arms_imports_tiv=(\"arms_imports_tiv\", \"sum\"))"
assert len(top15) == 15
assert list(top15["arms_imports_tiv"]) == sorted(top15["arms_imports_tiv"], reverse=True), "sort from large to small"
assert top15["arms_imports_tiv"].iloc[0] == totals["arms_imports_tiv"].max()
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### B2 Imports by regime over time

**Goal:** `by_regime`: total arms imports per year for each regime label, and a quick line chart.

#### What you need: one line per group in a chart

The week 5 pattern: loop over the groups and draw one line per group with `ax.plot(x, y, label=name)`, then `ax.legend()`.

*Seen in the lecture:* week 5, `lecture_week_5_data_viz.py` (`for name, g in df.groupby("country"): ax.plot(...)`)

In [ ]:
# Example: run this cell and look at the output
sales = pd.DataFrame({"shop": ["A", "A", "B", "B"], "year": [2020, 2021, 2020, 2021], "amount": [5, 7, 3, 6]})
fig, ax = plt.subplots()
for shop, rows in sales.groupby("shop"):
    ax.plot(rows["year"], rows["amount"], label=shop)
ax.legend()
plt.show()

#### Your turn

**Step 1.** Total `arms_imports_tiv` per `year` and `regime_label` (`as_index=False`), call it `by_regime`. Print the rows for 1990.

In [ ]:
# your code here

**Step 2.** Draw one line per regime label, like the example. Write one sentence on what you see.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert set(by_regime.columns) == {"year", "regime_label", "arms_imports_tiv"}
assert abs(by_regime["arms_imports_tiv"].sum() - panel["arms_imports_tiv"].sum()) < 0.01
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### B3 Democracy in war and peace

**Goal:** average democracy index in conflict years versus peace years.

#### Your turn

**Step 1.** Make two filtered tables `conflict_years` (`conflict_active == 1`) and `peace_years` (`conflict_active == 0`). Print the mean of `democracy_index` for each.

In [ ]:
# your code here

**Step 2.** Repeat only for wars (`max_intensity == 2`) versus peace. Is the gap larger? One sentence as a comment.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert (conflict_years["conflict_active"] == 1).all()
assert (peace_years["conflict_active"] == 0).all()
assert len(conflict_years) + len(peace_years) == len(panel)
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

---
## Part C: Chart functions (07)

Each function draws on an `ax` it receives. Test it here, then copy it unchanged into `07_charts.ipynb` and from there into `app/panels_arms.py` (C1 to C3) or `app/panels_analysis.py` (C4, C5).

**Rules for every timeline** (see `CONVENTIONS.md`): `ax.set_xlim(1950, 2025)`, a short title with `ax.set_title`, a y-axis label with the unit, and a dashed line `ax.axvline(year, linestyle="--", color="grey")` when `year` is given. Inside the function, use only the arguments (never a global table). The colour dictionaries at the top of the file are the one exception.

### C0 Load the files

**Goal:** the chart data loaded from the CSV files, as the app will do it.

#### What you need: drawing charts on an ax

`fig, ax = plt.subplots()` makes a figure (`fig`) with one drawing area (`ax`). You draw on `ax` with `ax.plot` (lines), `ax.bar` (vertical bars) or `ax.barh` (horizontal bars).

Our chart functions get the `ax` as their first argument and only draw on it. Then the app can put five charts under each other with `plt.subplots(5, 1, sharex=True)`, and they share the same year axis automatically.

Useful: `ax.set_xlim(1950, 2025)` fixes the year range, `ax.axvline(2014, linestyle="--")` draws a dashed vertical line, `ax.set_title("...")`, `ax.set_ylabel("...")`, `ax.text(0.5, 0.5, "message", transform=ax.transAxes, ha="center")` writes a message in the middle.

*Seen in the lecture:* week 5, page `lecture_week_5_data_viz.py` (`fig, ax = plt.subplots()`, `ax.plot`, `plt.tight_layout()`)

In [ ]:
# Example: run this cell and look at the output
def temperature_line(ax, df, city, year=None):
    rows = df[df["city"] == city]
    ax.plot(rows["year"], rows["temp"])
    ax.set_xlim(2000, 2005)
    ax.set_title("Temperature, " + city)
    ax.set_ylabel("°C")
    if year is not None:
        ax.axvline(year, linestyle="--", color="grey")

weather = pd.DataFrame({
    "city": ["Zurich"] * 5 + ["Bern"] * 5,
    "year": [2000, 2001, 2002, 2003, 2004] * 2,
    "temp": [9.1, 9.4, 9.0, 10.2, 9.8, 8.7, 8.9, 8.6, 9.9, 9.3],
})

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 4))
temperature_line(axes[0], weather, "Zurich", year=2003)
temperature_line(axes[1], weather, "Bern", year=2003)
plt.tight_layout()
plt.show()

#### Your turn

**Step 1.** Load `sipri_suppliers.csv` from `CLEAN` (`panel` is already loaded from Part B). Agree with Nicolas on the colour dictionaries (he writes them in his D0).

In [ ]:
# your code here

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### C1 arms_line(ax, panel, iso3, year=None)

**Goal:** arms imports over time for one country, as bars.

#### Your turn

**Step 1.** Without a function first: `fig, ax = plt.subplots(figsize=(10, 1.6))`, filter `panel` to `"IND"`, draw `ax.bar(rows["year"], rows["arms_imports_tiv"])`. Add `ax.set_xlim(1950, 2025)`, `ax.set_title("Arms imports")`, `ax.set_ylabel("TIV, millions")`. Show it.

In [ ]:
# your code here

**Step 2.** Add the dashed year line when `year` is given.

In [ ]:
# your code here

**Step 3.** Wrap it in `def arms_line(ax, panel, iso3, year=None):` (filter the argument `panel`) and test it with `"IND"` and 1999.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
fig, ax = plt.subplots()
arms_line(ax, panel, "IND", 1999)
assert ax.get_xlim() == (1950, 2025)
assert len(ax.patches) == len(panel[panel["iso3"] == "IND"]), "one bar per year"
assert len(ax.lines) == 1
fig, ax = plt.subplots()
only_ukr = panel[panel["iso3"] == "UKR"]
arms_line(ax, only_ukr, "IND")
assert len(ax.patches) == 0, "filter the argument panel, not the global table"
assert len(ax.lines) == 0
plt.close("all")
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### C2 supplier_bars(ax, suppliers, iso3, year=None)

**Goal:** top 3 suppliers + Other, as stacked bars by year.

#### What you need: stacked bars with pivot

For stacked bars we need one column per supplier: `pivot(index="year", columns="supplier", values="tiv")` turns the long table wide, and `.fillna(0)` fills years without deliveries. Then we draw one `ax.bar` per column, each on top of the previous ones with `bottom=`.

*Seen in the lecture:* week 4, `pivot` (presentation and Disney notebook)

In [ ]:
# Example: run this cell and look at the output
sales = pd.DataFrame({"year": [2020, 2020, 2021, 2022], "product": ["tea", "cake", "tea", "cake"], "amount": [5, 2, 4, 6]})
wide = sales.pivot(index="year", columns="product", values="amount").fillna(0)
print(wide)

fig, ax = plt.subplots(figsize=(6, 2))
bottom = [0] * len(wide)
for product in wide.columns:
    ax.bar(wide.index, wide[product], bottom=bottom, label=product)
    bottom = [b + v for b, v in zip(bottom, wide[product])]
ax.legend()
plt.show()

#### Your turn

**Step 1.** Filter `sipri_suppliers` to one country and `pivot` it like the example.

In [ ]:
# your code here

**Step 2.** Draw the stacked bars, then add the timeline rules (title `"Suppliers"`, y-label `"TIV, millions"`, xlim, dashed year line) and a small legend: `ax.legend(fontsize=7, loc="upper left")`.

In [ ]:
# your code here

**Step 3.** A country with no imports: if the filtered table is empty, write `"No arms imports recorded"` with `ax.text(0.5, 0.5, "...", transform=ax.transAxes, ha="center")` (see the C0 box above) and skip the bars. Wrap everything in the function, using the argument `suppliers`.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
fig, ax = plt.subplots()
supplier_bars(ax, sipri_suppliers, "XXX")
assert len(ax.texts) == 1, "show a message when there is no data"
fig, ax = plt.subplots()
only_ind = sipri_suppliers[sipri_suppliers["iso3"] == "IND"]
supplier_bars(ax, only_ind, "SAU")
assert len(ax.texts) == 1 and len(ax.patches) == 0, "filter the argument suppliers, not the global table"
fig, ax = plt.subplots()
supplier_bars(ax, sipri_suppliers, "IND", year=1999)
assert len(ax.patches) > 0 and ax.get_xlim() == (1950, 2025) and len(ax.lines) == 1
plt.close("all")
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### C3 global_trends(ax, panel, year=None)

**Goal:** the world over time for the Overview page: the share of countries in conflict per year.

#### Your turn

**Step 1.** Without a function first: per `year`, the mean of `conflict_active` (`groupby("year", as_index=False).agg(share=("conflict_active", "mean"))`). Draw it with `ax.plot`.

In [ ]:
# your code here

**Step 2.** Timeline rules: xlim, title `"Share of countries in conflict"`, y-label `"share"`, dashed year line.

In [ ]:
# your code here

**Step 3.** Wrap it in `def global_trends(ax, panel, year=None):`.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
fig, ax = plt.subplots()
global_trends(ax, panel, 2000)
assert len(ax.lines) == 2, "one line for the share and one dashed year line"
assert ax.get_xlim() == (1950, 2025)
plt.close("all")
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### C4 democracy_line(ax, panel, iso3, year=None)

**Goal:** democracy index over time for one country.

#### Your turn

**Step 1.** Filter `panel` to one country and draw `democracy_index` over `year` with `ax.plot`.

In [ ]:
# your code here

**Step 2.** Fix the y-axis from 0 to 1 with `ax.set_ylim(0, 1)`, add the timeline rules (title `"Democracy index"`, y-label `"0 to 1"`), the dashed year line, then wrap it in the function.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
fig, ax = plt.subplots()
democracy_line(ax, panel, "UKR", year=2014)
assert ax.get_xlim() == (1950, 2025) and ax.get_ylim() == (0, 1)
assert len(ax.lines) == 2, "the democracy line and the dashed year line"
fig, ax = plt.subplots()
only_ind = panel[panel["iso3"] == "IND"]
democracy_line(ax, only_ind, "UKR")
assert len(ax.lines[0].get_xdata()) == 0, "filter the argument panel, not the global table"
plt.close("all")
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### C5 main_chart(ax, event_time)

**Goal:** the main result: `dem_change` by `t`, one line per importer group.

#### Your turn

**Step 1.** Load `event_time.csv` (made together in 06 Part B). Print it.

In [ ]:
# your code here

**Step 2.** Draw one line per `importer_group`, like the example in B2.

In [ ]:
# your code here

**Step 3.** Add `ax.axvline(0, color="grey")` (conflict onset), `ax.axhline(0, color="grey", linewidth=0.8)`, axis labels "Years since conflict onset" and "Change in democracy index", a legend, and wrap it in the function.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
event_time = pd.read_csv(CLEAN + "event_time.csv")
fig, ax = plt.subplots()
main_chart(ax, event_time)
assert len(ax.lines) == 4, "two group lines, one vertical and one horizontal line"
assert ax.get_legend() is not None, "add a legend"
plt.close("all")
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

## Before you move code into the main notebook

- [ ] Kernel → Restart and Run All runs this sandbox from top to bottom without an error
- [ ] All checkpoints above print OK
- [ ] Names follow `CONVENTIONS.md`
- [ ] You wrote the "in your own words" sentence for every task
- [ ] You can explain every line to your partner without looking at the examples
- [ ] Copy only your working cells into the numbered main notebook (no examples, no experiments)
- [ ] Run the main notebook with Restart and Run All, then commit and push